In [3]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler
import pickle
import os

# Load and preprocess data
def load_and_preprocess_data():
    try:
        d1 = pd.read_csv('gran/0_6_token_counts.csv', index_col='Unnamed: 0')
        d2 = pd.read_csv('gran/6_12_token_counts.csv', index_col='Unnamed: 0')
        d3 = pd.read_csv('gran/12_18_token_counts.csv', index_col='Unnamed: 0')
        d4 = pd.read_csv('gran/18_24_token_counts.csv', index_col='Unnamed: 0')
    except FileNotFoundError as e:
        raise FileNotFoundError(f"CSV file not found: {e}")

    common_index = d1.index.intersection(d2.index).intersection(d3.index).intersection(d4.index)
    if len(common_index) == 0:
        raise ValueError("No common indices found across CSV files.")
    d1, d2, d3, d4 = d1.loc[common_index], d2.loc[common_index], d3.loc[common_index], d4.loc[common_index]

    data = np.stack([d1.values, d2.values, d3.values, d4.values], axis=1)
    actual_input_dim = data.shape[-1]
    expected_input_dim = 129
    if actual_input_dim != expected_input_dim:
        print(f"Warning: Actual input_dim={actual_input_dim}, expected {expected_input_dim}. Using actual dimension.")

    data = np.nan_to_num(data, nan=0.0)
    scaler = StandardScaler()
    data_reshaped = data.reshape(-1, data.shape[-1])
    data_normalized = scaler.fit_transform(data_reshaped).reshape(data.shape)

    return data_normalized, scaler, actual_input_dim

# Placeholder for covariance-based feature selection
def compute_covariance(data, G, b):
    return np.cov(data.reshape(-1, data.shape[-1]).T).mean()

def select_best_scheme(data, granularities=[1, 2, 3, 4, 6, 8, 12, 24]):
    best_G, best_b, max_cov = None, None, -np.inf
    for G in granularities:
        for b in range(G):
            cov = compute_covariance(data, G, b)
            if cov > max_cov:
                max_cov, best_G, best_b = cov, G, b
    return best_G, best_b

# Encoder class
class Encoder(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_layers, latent_dim):
        super(Encoder, self).__init__()
        self.bilstm = nn.LSTM(input_dim, hidden_dim, num_layers, batch_first=True, bidirectional=True)
        self.fc = nn.Sequential(
            nn.Linear(hidden_dim * 2, 128),
            nn.ReLU(),
            nn.Linear(128, latent_dim)
        )

    def forward(self, x):
        lstm_out, _ = self.bilstm(x)
        latent = self.fc(torch.mean(lstm_out, dim=1))
        return latent

# Decoder class
class Decoder(nn.Module):
    def __init__(self, latent_dim, output_dim, seq_len):
        super(Decoder, self).__init__()
        self.seq_len = seq_len
        self.output_dim = output_dim
        self.fc = nn.Sequential(
            nn.Linear(latent_dim, 128),
            nn.ReLU(),
            nn.Linear(128, output_dim * seq_len)
        )

    def forward(self, x):
        reconstructed = self.fc(x)
        reconstructed = reconstructed.view(-1, self.seq_len, self.output_dim)
        return reconstructed

# Autoencoder class
class Autoencoder(nn.Module):
    def __init__(self, encoder, decoder):
        super(Autoencoder, self).__init__()
        self.encoder = encoder
        self.decoder = decoder

    def forward(self, x):
        latent = self.encoder(x)
        reconstructed = self.decoder(latent)
        return reconstructed

# Training function
def train_autoencoder(model, train_loader, criterion, optimizer, epochs, device):
    model.train()
    losses = []
    for epoch in range(epochs):
        epoch_loss = 0
        for batch in train_loader:
            inputs = batch[0].to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, inputs)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()
        avg_loss = epoch_loss / len(train_loader)
        losses.append(avg_loss)
        print(f"Epoch [{epoch+1}/{epochs}], Loss: {avg_loss:.4f}")
    return losses

# Evaluate reconstruction errors
def evaluate_autoencoder(model, data_tensor, criterion, device):
    model.eval()
    reconstruction_errors = []
    with torch.no_grad():
        for i in range(len(data_tensor)):
            input_data = data_tensor[i:i+1].to(device)
            output = model(input_data)
            error = criterion(output, input_data).item()
            reconstruction_errors.append(error)
    return reconstruction_errors

# Detect anomalies
def detect_anomalies(reconstruction_errors, threshold=None):
    if threshold is None:
        threshold = np.mean(reconstruction_errors) + 2 * np.std(reconstruction_errors)
    anomalies = [i for i, error in enumerate(reconstruction_errors) if error > threshold]
    return anomalies, threshold

# Main function
def main():
    # Parameters (from paper)
    expected_input_dim = 129
    seq_len = 4
    hidden_dim = 64
    num_layers = 3
    latent_dim = 32
    batch_size = 16
    epochs = 500
    learning_rate = 0.001

    # Device
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")

    # Load and preprocess data
    data_normalized, scaler, actual_input_dim = load_and_preprocess_data()
    input_dim = actual_input_dim
    output_dim = input_dim

    # User-adaptive feature selection (placeholder)
    best_G, best_b = select_best_scheme(data_normalized)
    print(f"Best granularity: {best_G}h, Best bias: {best_b}h")

    # Convert to tensor
    data_tensor = torch.tensor(data_normalized, dtype=torch.float32)

    # Create DataLoader
    dataset = TensorDataset(data_tensor)
    train_loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

    # Initialize model
    encoder = Encoder(input_dim, hidden_dim, num_layers, latent_dim)
    decoder = Decoder(latent_dim, output_dim, seq_len)
    autoencoder = Autoencoder(encoder, decoder).to(device)

    # Loss and optimizer
    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(autoencoder.parameters(), lr=learning_rate)

    # Train model
    losses = train_autoencoder(autoencoder, train_loader, criterion, optimizer, epochs, device)

    # Save model, scaler, and metadata
    save_dir = "."
    os.makedirs(save_dir, exist_ok=True)

    # Save model state as britd_model.pth
    model_path = os.path.join(save_dir, "britd_model.pth")
    torch.save(autoencoder.state_dict(), model_path)
    print(f"Model saved to {model_path}")

    # Save scaler
    scaler_path = os.path.join(save_dir, "scaler.pkl")
    with open(scaler_path, "wb") as f:
        pickle.dump(scaler, f)
    print(f"Scaler saved to {scaler_path}")

    # Save metadata
    metadata = {
        "input_dim": input_dim,
        "seq_len": seq_len,
        "hidden_dim": hidden_dim,
        "num_layers": num_layers,
        "latent_dim": latent_dim
    }
    metadata_path = os.path.join(save_dir, "metadata.pkl")
    with open(metadata_path, "wb") as f:
        pickle.dump(metadata, f)
    print(f"Metadata saved to {metadata_path}")

    # Evaluate reconstruction errors
    reconstruction_errors = evaluate_autoencoder(autoencoder, data_tensor, criterion, device)

    # Detect anomalies
    anomalies, threshold = detect_anomalies(reconstruction_errors)
    print(f"\nDetected {len(anomalies)} anomalies with threshold {threshold:.4f}")
    print(f"Top 5 anomalies (indices): {anomalies[:5]}")

    # Example: Reconstruct a sample
    sample_idx = 0
    autoencoder.eval()
    with torch.no_grad():
        input_sample = data_tensor[sample_idx:sample_idx+1].to(device)
        reconstructed = autoencoder(input_sample)
        print(f"\nOriginal sample (normalized, first 5 features): {input_sample[0, 0, :5]}")
        print(f"Reconstructed sample (first 5 features): {reconstructed[0, 0, :5]}")

if __name__ == "__main__":
    main()

Using device: cuda
Best granularity: 1h, Best bias: 0h
Epoch [1/500], Loss: 0.6869
Epoch [2/500], Loss: 0.5100
Epoch [3/500], Loss: 0.3839
Epoch [4/500], Loss: 0.3352
Epoch [5/500], Loss: 0.3175
Epoch [6/500], Loss: 0.3183
Epoch [7/500], Loss: 0.3113
Epoch [8/500], Loss: 0.3002
Epoch [9/500], Loss: 0.2936
Epoch [10/500], Loss: 0.2868
Epoch [11/500], Loss: 0.2714
Epoch [12/500], Loss: 0.2620
Epoch [13/500], Loss: 0.2631
Epoch [14/500], Loss: 0.2515
Epoch [15/500], Loss: 0.2357
Epoch [16/500], Loss: 0.2312
Epoch [17/500], Loss: 0.2266
Epoch [18/500], Loss: 0.2257
Epoch [19/500], Loss: 0.2129
Epoch [20/500], Loss: 0.2058
Epoch [21/500], Loss: 0.1949
Epoch [22/500], Loss: 0.1891
Epoch [23/500], Loss: 0.1943
Epoch [24/500], Loss: 0.1821
Epoch [25/500], Loss: 0.1679
Epoch [26/500], Loss: 0.1679
Epoch [27/500], Loss: 0.1555
Epoch [28/500], Loss: 0.1460
Epoch [29/500], Loss: 0.1491
Epoch [30/500], Loss: 0.1489
Epoch [31/500], Loss: 0.1390
Epoch [32/500], Loss: 0.1495
Epoch [33/500], Loss: 0.13